# RAG Agent Consolidated Notebook

This notebook consolidates the full pipeline for the macro-aware credit risk RAG agent.

Pipeline order:
1. Metadata Generation
2. Text Extraction
3. Chunking
4. Embeddings and Retrieval
5. LLM / Agent Generation

Source notebooks were merged in the order requested.

In [ ]:
%pip install pymupdf
%pip install sentence-transformers faiss-cpu pandas numpy

In [11]:
from pathlib import Path

# Base directory = where this notebook is located
BASE_DIR = Path().resolve()

# Define all folders relative to notebook
RAW_REPORTS_DIR = BASE_DIR / "RAW Reports"
METADATA_DIR = BASE_DIR / "Metadata"
CLEANED_DIR = BASE_DIR / "Cleaned Reports"
CHUNKS_DIR = BASE_DIR / "Chunks"
EMBEDDINGS_DIR = BASE_DIR / "Embeddings"

print("Base directory:", BASE_DIR)

Base directory: C:\Users\Achal Agarwal\OneDrive - The University of Hong Kong\HKU Shiz\Year 3\Sem 2\COMP3520\Group Project\Project Implementation\RAG Agent


## Metadata Code

In [12]:
import pandas as pd
import os

folder = RAW_REPORTS_DIR
output_path = METADATA_DIR / "report_metadata.csv"
rows = []

for f in os.listdir(folder):
    if f.endswith(".pdf"):
        parts = f.replace(".pdf", "").split("_")
        rows.append({
            "doc_id": f.replace(".pdf", ""),
            "filename": f,
            "year": int(parts[1]),
            "month": int(parts[2]),
            "source": "Federal Reserve"
        })

df = pd.DataFrame(rows)

# Save to Metadata folder
df.to_csv(output_path, index=False)

print(df.head())

        doc_id         filename  year  month           source
0  mpr_2007_02  mpr_2007_02.pdf  2007      2  Federal Reserve
1  mpr_2007_07  mpr_2007_07.pdf  2007      7  Federal Reserve
2  mpr_2008_02  mpr_2008_02.pdf  2008      2  Federal Reserve
3  mpr_2008_07  mpr_2008_07.pdf  2008      7  Federal Reserve
4  mpr_2009_02  mpr_2009_02.pdf  2009      2  Federal Reserve


---

## Text-Extraction

In [13]:
import os
import fitz  # PyMuPDF

input_folder = RAW_REPORTS_DIR
output_folder = CLEANED_DIR
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if not filename.endswith(".pdf"):
        continue

    pdf_path = os.path.join(input_folder, filename)
    txt_filename = filename.replace(".pdf", ".txt")
    output_path = os.path.join(output_folder, txt_filename)

    full_text = []

    try:
        doc = fitz.open(pdf_path)

        for page_num, page in enumerate(doc, start=1):
            text = page.get_text("text")
            if text:
                full_text.append(f"\n--- Page {page_num} ---\n")
                full_text.append(text)

        with open(output_path, "w", encoding="utf-8") as f:
            f.write("\n".join(full_text))

        print(f"✅ Saved: {txt_filename}")

    except Exception as e:
        print(f"❌ Error processing {filename}: {e}")

✅ Saved: mpr_2007_02.txt
✅ Saved: mpr_2007_07.txt
✅ Saved: mpr_2008_02.txt
✅ Saved: mpr_2008_07.txt
✅ Saved: mpr_2009_02.txt
✅ Saved: mpr_2009_07.txt
✅ Saved: mpr_2010_02.txt
✅ Saved: mpr_2010_07.txt
✅ Saved: mpr_2011_03.txt
✅ Saved: mpr_2011_07.txt
✅ Saved: mpr_2012_02.txt
✅ Saved: mpr_2012_07.txt
✅ Saved: mpr_2013_02.txt
✅ Saved: mpr_2013_07.txt
✅ Saved: mpr_2014_02.txt
✅ Saved: mpr_2014_07.txt
✅ Saved: mpr_2015_02.txt
✅ Saved: mpr_2015_07.txt
✅ Saved: mpr_2016_02.txt
✅ Saved: mpr_2016_06.txt
✅ Saved: mpr_2017_02.txt
✅ Saved: mpr_2017_07.txt
✅ Saved: mpr_2018_02.txt
✅ Saved: mpr_2018_07.txt
✅ Saved: mpr_2019_02.txt
✅ Saved: mpr_2019_07.txt
✅ Saved: mpr_2020_02.txt
✅ Saved: mpr_2020_06.txt
✅ Saved: mpr_2021_02.txt
✅ Saved: mpr_2021_07.txt
✅ Saved: mpr_2022_02.txt
✅ Saved: mpr_2022_06.txt
✅ Saved: mpr_2023_03.txt
✅ Saved: mpr_2023_06.txt
✅ Saved: mpr_2024_03.txt
✅ Saved: mpr_2024_07.txt
✅ Saved: mpr_2025_02.txt
✅ Saved: mpr_2025_06.txt


In [14]:
import os
import re

folder = CLEANED_DIR

for filename in os.listdir(folder):
    if not filename.endswith(".txt"):
        continue

    path = os.path.join(folder, filename)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    # Fix common ligatures
    text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")

    # Normalize spaces
    text = re.sub(r"[ \t]+", " ", text)

    # Normalize blank lines
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    with open(path, "w", encoding="utf-8") as f:
        f.write(text.strip())

    print(f"Cleaned: {filename}")

Cleaned: mpr_2007_02.txt
Cleaned: mpr_2007_07.txt
Cleaned: mpr_2008_02.txt
Cleaned: mpr_2008_07.txt
Cleaned: mpr_2009_02.txt
Cleaned: mpr_2009_07.txt
Cleaned: mpr_2010_02.txt
Cleaned: mpr_2010_07.txt
Cleaned: mpr_2011_03.txt
Cleaned: mpr_2011_07.txt
Cleaned: mpr_2012_02.txt
Cleaned: mpr_2012_07.txt
Cleaned: mpr_2013_02.txt
Cleaned: mpr_2013_07.txt
Cleaned: mpr_2014_02.txt
Cleaned: mpr_2014_07.txt
Cleaned: mpr_2015_02.txt
Cleaned: mpr_2015_07.txt
Cleaned: mpr_2016_02.txt
Cleaned: mpr_2016_06.txt
Cleaned: mpr_2017_02.txt
Cleaned: mpr_2017_07.txt
Cleaned: mpr_2018_02.txt
Cleaned: mpr_2018_07.txt
Cleaned: mpr_2019_02.txt
Cleaned: mpr_2019_07.txt
Cleaned: mpr_2020_02.txt
Cleaned: mpr_2020_06.txt
Cleaned: mpr_2021_02.txt
Cleaned: mpr_2021_07.txt
Cleaned: mpr_2022_02.txt
Cleaned: mpr_2022_06.txt
Cleaned: mpr_2023_03.txt
Cleaned: mpr_2023_06.txt
Cleaned: mpr_2024_03.txt
Cleaned: mpr_2024_07.txt
Cleaned: mpr_2025_02.txt
Cleaned: mpr_2025_06.txt


---

## Chunks

In [15]:
import os
import re
import json
import pandas as pd

input_folder = CLEANED_DIR
output_folder = CHUNKS_DIR

os.makedirs(output_folder, exist_ok=True)

# ----------------------------
# Helpers
# ----------------------------

def word_count(text):
    return len(text.split())

def normalize_text(text):
    text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    return text.strip()

def parse_metadata_from_filename(filename):
    name = filename.replace(".txt", "")
    parts = name.split("_")
    return {
        "doc_id": name,
        "year": int(parts[1]),
        "month": int(parts[2]),
    }

def is_page_marker(line):
    return bool(re.match(r"^--- Page \d+ ---$", line.strip()))

def is_noise_line(line):
    line = line.strip()

    if not line:
        return True

    if is_page_marker(line):
        return True

    if re.fullmatch(r"[ivxIVX\d]+", line):
        return True

    if line.startswith("NOTE:") or line.startswith("SOURCE:"):
        return True

    if line in {
        "Monetary Policy Report",
        "to the Congress",
        "Board of Governors of the Federal Reserve System",
        "Submitted pursuant to section 2B",
        "of the Federal Reserve Act",
        "Letter of Transmittal",
        "Contents",
        "Boxes",
        "Abbreviations",
    }:
        return True

    # chart/axis style short lines
    if word_count(line) <= 6:
        if re.search(r"Percent|Billions|Millions|Index|annual rate", line):
            return True
        if sum(c.isdigit() for c in line) >= 2:
            return True

    # figure number style
    if re.match(r"^\d+\.\s", line):
        return True

    return False

def is_contents_line(line):
    line = line.strip()
    contents_signals = [
        "Part 1", "Part 2", "Part 3", "Part 4",
        "Overview:", "Recent Economic and Financial Developments",
        "Financial Developments", "International Developments"
    ]
    return any(sig in line for sig in contents_signals)

def is_major_section(line):
    line = line.strip()

    if re.match(r"^Part \d+$", line):
        return True

    major_sections = {
        "Overview:",
        "Recent Economic and Financial Developments",
        "Domestic Developments",
        "Financial Developments",
        "International Developments",
        "The Household Sector",
        "The Business Sector",
        "The Government Sector",
        "The External Sector",
        "The Labor Market",
        "Monetary Policy",
        "Summary of Economic Projections",
    }

    return line in major_sections

def is_subsection(line):
    line = line.strip()

    if not line:
        return False

    if is_major_section(line):
        return False

    banned = {
        "Page",
        "Monetary Policy Report",
        "to the Congress",
        "Board of Governors of the Federal Reserve System",
        "For use at 10:00 a.m., EST",
        "For use at 2:00 p.m., EDT",
        "Part 1", "Part 2", "Part 3", "Part 4"
    }
    if line in banned:
        return False

    # likely subsection: short title-case line
    words = line.split()
    if 2 <= len(words) <= 10:
        if line.endswith("."):
            return False
        alpha_words = [w for w in words if re.search(r"[A-Za-z]", w)]
        if alpha_words and all(
            w[:1].isupper() or w.isupper() or "’" in w or "-" in w
            for w in alpha_words
        ):
            return True

    return False

# ----------------------------
# Line-based parser
# ----------------------------

def extract_structured_blocks(text, metadata):
    text = normalize_text(text)
    lines = [ln.strip() for ln in text.split("\n")]

    current_section = None
    current_subsection = None
    current_paragraph = []
    blocks = []

    in_contents = False

    def flush_paragraph():
        nonlocal current_paragraph
        if current_paragraph:
            para = " ".join(current_paragraph).strip()
            para = normalize_text(para)
            if word_count(para) >= 25:
                blocks.append({
                    "section": current_section if current_section else "Uncategorized",
                    "subsection": current_subsection if current_subsection else "General",
                    "text": para
                })
            current_paragraph = []

    for line in lines:
        if not line:
            flush_paragraph()
            continue

        if is_page_marker(line):
            flush_paragraph()
            continue

        if is_noise_line(line):
            continue

        # detect contents section and skip it
        if line == "Contents":
            flush_paragraph()
            in_contents = True
            continue

        if in_contents:
            # stop skipping contents once we hit real body content
            if line.startswith("Part 1") or line == "Overview:":
                in_contents = False
            else:
                if is_contents_line(line) or re.match(r"^\d+", line):
                    continue

        if is_major_section(line):
            flush_paragraph()
            current_section = line
            current_subsection = None
            continue

        if is_subsection(line):
            flush_paragraph()
            current_subsection = line
            continue

        current_paragraph.append(line)

    flush_paragraph()
    return blocks

def merge_blocks_into_chunks(blocks, metadata, min_words=180, max_words=380):
    chunks = []
    chunk_idx = 1

    buffer = []
    buffer_words = 0
    current_section = None
    current_subsection = None

    def flush_buffer():
        nonlocal buffer, buffer_words, chunk_idx, current_section, current_subsection
        if not buffer:
            return
        chunk_text = "\n\n".join(buffer).strip()
        if word_count(chunk_text) >= 40:
            chunks.append({
                "chunk_id": f"{metadata['doc_id']}_chunk_{chunk_idx:03d}",
                "doc_id": metadata["doc_id"],
                "year": metadata["year"],
                "month": metadata["month"],
                "section": current_section if current_section else "Uncategorized",
                "subsection": current_subsection if current_subsection else "General",
                "word_count": word_count(chunk_text),
                "text": chunk_text
            })
            chunk_idx += 1
        buffer = []
        buffer_words = 0

    for block in blocks:
        block_section = block["section"]
        block_subsection = block["subsection"]
        block_text = block["text"]
        block_words = word_count(block_text)

        # flush if section/subsection changes
        if buffer and (
            block_section != current_section or block_subsection != current_subsection
        ):
            flush_buffer()

        if not buffer:
            current_section = block_section
            current_subsection = block_subsection

        # flush if chunk would get too large
        if buffer and (buffer_words + block_words > max_words):
            flush_buffer()
            current_section = block_section
            current_subsection = block_subsection

        buffer.append(block_text)
        buffer_words += block_words

    flush_buffer()
    return chunks

def process_document(text, metadata):
    blocks = extract_structured_blocks(text, metadata)
    chunks = merge_blocks_into_chunks(blocks, metadata)
    return chunks

# ----------------------------
# Run
# ----------------------------

all_chunks = []

for filename in sorted(os.listdir(input_folder)):
    if not filename.endswith(".txt"):
        continue

    metadata = parse_metadata_from_filename(filename)
    path = os.path.join(input_folder, filename)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    doc_chunks = process_document(text, metadata)
    all_chunks.extend(doc_chunks)

df_chunks = pd.DataFrame(all_chunks)

csv_path = os.path.join(output_folder, "monetary_policy_chunks_v3.csv")
jsonl_path = os.path.join(output_folder, "monetary_policy_chunks_v3.jsonl")

df_chunks.to_csv(csv_path, index=False, encoding="utf-8")

with open(jsonl_path, "w", encoding="utf-8") as f:
    for row in all_chunks:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Saved {len(df_chunks)} chunks.")
print(csv_path)
print(jsonl_path)
print(df_chunks.head(10))

Saved 2644 chunks.
C:\Users\Achal Agarwal\OneDrive - The University of Hong Kong\HKU Shiz\Year 3\Sem 2\COMP3520\Group Project\Project Implementation\RAG Agent\Chunks\monetary_policy_chunks_v3.csv
C:\Users\Achal Agarwal\OneDrive - The University of Hong Kong\HKU Shiz\Year 3\Sem 2\COMP3520\Group Project\Project Implementation\RAG Agent\Chunks\monetary_policy_chunks_v3.jsonl
                chunk_id       doc_id  year  month        section  \
0  mpr_2007_02_chunk_001  mpr_2007_02  2007      2  Uncategorized   
1  mpr_2007_02_chunk_002  mpr_2007_02  2007      2  Uncategorized   
2  mpr_2007_02_chunk_003  mpr_2007_02  2007      2  Uncategorized   
3  mpr_2007_02_chunk_004  mpr_2007_02  2007      2  Uncategorized   
4  mpr_2007_02_chunk_005  mpr_2007_02  2007      2  Uncategorized   
5  mpr_2007_02_chunk_006  mpr_2007_02  2007      2  Uncategorized   
6  mpr_2007_02_chunk_007  mpr_2007_02  2007      2  Uncategorized   
7  mpr_2007_02_chunk_008  mpr_2007_02  2007      2  Uncategorized   
8  m

---

## Embed and retrieval

In [18]:
import os
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Paths
chunk_file = CHUNKS_DIR / "monetary_policy_chunks_v3.csv"
output_folder = EMBEDDINGS_DIR
os.makedirs(output_folder, exist_ok=True)

faiss_path = os.path.join(output_folder, "monetary_policy_faiss.index")
metadata_path = os.path.join(output_folder, "monetary_policy_metadata.csv")

# Load chunks
df = pd.read_csv(chunk_file)
texts = df["text"].fillna("").tolist()

# Embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Create embeddings
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# Save
faiss.write_index(index, str(FAISS_INDEX_PATH))
df.to_csv(str(METADATA_CSV_PATH), index=False)

print("Saved index and metadata.")

# Retrieval function
def retrieve_chunks_by_year(query, embedding_model, index, df_meta, year, k=3):

    # Filter metadata for that year
    df_filtered = df_meta[df_meta["year"] == year].reset_index(drop=True)

    if len(df_filtered) == 0:
        print(f"No data found for year {year}")
        return []

    # Get indices of filtered rows
    filtered_indices = df_filtered.index.values

    # Embed query
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)

    # Search full index
    distances, indices = index.search(query_embedding, k * 5)

    # Keep only results from the selected year
    results = []
    for idx in indices[0]:
        if idx in filtered_indices:
            row = df_meta.iloc[idx]
            results.append({
                "section": row["section"],
                "subsection": row["subsection"],
                "text": row["text"]
            })
        if len(results) >= k:
            break

    return results

# Test
query = "How does unemployment in affect household financial stress and default risk?"
results = retrieve_chunks(query, model, index, df, k=5)

for r in results:
    print("\n" + "="*80)
    print("Rank:", r["rank"])
    print("Section:", r["section"])
    print("Subsection:", r["subsection"])
    print("Distance:", r["distance"])
    print(r["text"][:600])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Saved index and metadata.

Rank: 1
Section: Summary of Economic Projections
Subsection: The Outlook
Distance: 0.8481864929199219
infl uenced by their judgments about the measured rates of infl ation consistent with the Federal Reserve’s dual mandate to promote maximum employment and price stability and about the time frame over which policy should aim to attain those rates given current economic conditions. Many participants judged that, given the recent adverse shocks to both aggregate demand and infl ation, policy would be able to foster only a gradual return of key macroeconomic variables to their longer- run sustainable or optimal levels. Consequently, the rate of unemployment was projected by some partici- pants to

Rank: 2
Section: Domestic Developments
Subsection: General
Distance: 0.9146506786346436
Part 1:   Recent Economic and Financial Developments improve overall, the financial system has not fully recovered from the financial crisis, and banks remained cautious in their le

---

## LLM

In [19]:
import json
import os
from typing import Any

import faiss
import numpy as np
import pandas as pd
import requests
from sentence_transformers import SentenceTransformer


# =========================================================
# CONFIG
# =========================================================

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
OLLAMA_MODEL_NAME = "phi3:mini"

FAISS_INDEX_PATH = EMBEDDINGS_DIR / "monetary_policy_faiss.index"
METADATA_CSV_PATH = EMBEDDINGS_DIR / "monetary_policy_metadata.csv"

TOP_K = 5
OLLAMA_URL = "http://localhost:11434/api/generate"
REQUEST_TIMEOUT = 180


# =========================================================
# FEATURE LISTS
# =========================================================

BORROWER_FEATURES = [
    "annual_inc",
    "dti",
    "fico_score",
    "loan_amnt",
    "term",
    "int_rate",
    "emp_length",
]

MACRO_FEATURES = [
    "unemployment_rate",
    "cpi",
    "fed_funds_rate",
    "yield_curve_spread",
    "gdp_growth",
]


# =========================================================
# LOAD MODELS / DATA
# =========================================================

def load_resources() -> tuple[SentenceTransformer, Any, pd.DataFrame]:
    """Load embedding model, FAISS index, and chunk metadata."""
    print("Loading embedding model...")
    embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    if not os.path.exists(FAISS_INDEX_PATH):
        raise FileNotFoundError(f"FAISS index not found: {FAISS_INDEX_PATH}")

    if not os.path.exists(METADATA_CSV_PATH):
        raise FileNotFoundError(f"Metadata CSV not found: {METADATA_CSV_PATH}")

    print("Loading FAISS index...")
    index = faiss.read_index(str(FAISS_INDEX_PATH))

    print("Loading metadata CSV...")
    df_meta = pd.read_csv(str(METADATA_CSV_PATH))

    required_cols = {"chunk_id", "section", "subsection", "text"}
    missing = required_cols - set(df_meta.columns)
    if missing:
        raise ValueError(f"Metadata CSV is missing required columns: {missing}")

    if index.ntotal != len(df_meta):
        raise ValueError(
            f"Mismatch: FAISS index has {index.ntotal} vectors, "
            f"but metadata has {len(df_meta)} rows."
        )

    print("Resources loaded successfully.")
    return embedding_model, index, df_meta

import re

def normalize_variable_name(value: str | None) -> str | None:
    if value is None:
        return None

    v = value.strip().lower()

    alias_map = {
        "dti": "dti",
        "dti (debt-to-income) ratio": "dti",
        "debt-to-income ratio": "dti",

        "annual_inc": "annual_inc",
        "annual income": "annual_inc",
        "annual income (annual_inc)": "annual_inc",

        "fico_score": "fico_score",
        "fico score": "fico_score",
        "current credit score": "fico_score",

        "loan_amnt": "loan_amnt",
        "loan amount": "loan_amnt",
        "loan amount (loan_amnt)": "loan_amnt",

        "term": "term",
        "loan term": "term",
        "loan term (term)": "term",

        "int_rate": "int_rate",
        "interest rate": "int_rate",
        "interest rate (int_rate)": "int_rate",

        "emp_length": "emp_length",
        "employment length": "emp_length",

        "unemployment_rate": "unemployment_rate",
        "unemployment rate": "unemployment_rate",
        "current unemployment rate": "unemployment_rate",

        "cpi": "cpi",
        "consumer price index": "cpi",
        "consumer price index (cpi)": "cpi",

        "fed_funds_rate": "fed_funds_rate",
        "federal funds rate": "fed_funds_rate",
        "federal funds rate (fed_funds_rate)": "fed_funds_rate",

        "yield_curve_spread": "yield_curve_spread",
        "yield curve spread": "yield_curve_spread",

        "gdp_growth": "gdp_growth",
        "gdp growth": "gdp_growth",
        "economic growth rate": "gdp_growth",
    }

    if v in alias_map:
        return alias_map[v]

    # fallback: extract text inside parentheses, e.g. "Annual Income (annual_inc)"
    m = re.search(r"\(([^)]+)\)", v)
    if m:
        inside = m.group(1).strip().lower()
        if inside in alias_map:
            return alias_map[inside]
        return inside

    return v


def validate_suggestions(suggestions, borrower_features, macro_features):
    valid = []

    for s in suggestions:
        borrower_raw = s.get("borrower_variable")
        macro_raw = s.get("macro_variable")

        borrower = normalize_variable_name(borrower_raw)
        macro = normalize_variable_name(macro_raw)

        if borrower not in borrower_features:
            continue
        if macro not in macro_features:
            continue

        s["borrower_variable"] = borrower
        s["macro_variable"] = macro
        s["interaction_feature"] = f"{borrower}_x_{macro}"

        # fill missing rationale if model omitted it
        if "rationale" not in s or not s["rationale"]:
            s["rationale"] = "Feature suggested from retrieved macroeconomic context."

        valid.append(s)

    return valid

def deduplicate_suggestions(validated_output):
    deduped = {}
    for item in validated_output:
        key = item["interaction_feature"]
        if key not in deduped:
            deduped[key] = item
    return list(deduped.values())

# =========================================================
# RETRIEVAL
# =========================================================

def retrieve_chunks(
    query: str,
    embedding_model: SentenceTransformer,
    index: Any,
    df_meta: pd.DataFrame,
    k: int = TOP_K,
) -> list[dict[str, Any]]:
    """Retrieve top-k chunks for a query."""
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
    ).astype("float32")

    distances, indices = index.search(query_embedding, k)

    results: list[dict[str, Any]] = []
    for rank, idx in enumerate(indices[0]):
        row = df_meta.iloc[idx]
        results.append(
            {
                "rank": rank + 1,
                "chunk_id": row["chunk_id"],
                "section": row["section"],
                "subsection": row["subsection"],
                "text": row["text"],
                "distance": float(distances[0][rank]),
            }
        )

    return results


def print_retrieval_results(results: list[dict[str, Any]], preview_chars: int = 500) -> None:
    """Pretty-print retrieved chunks."""
    for r in results:
        print("\n" + "=" * 100)
        print(f"Rank: {r['rank']}")
        print(f"Chunk ID: {r['chunk_id']}")
        print(f"Section: {r['section']}")
        print(f"Subsection: {r['subsection']}")
        print(f"Distance: {r['distance']:.4f}")
        print("Text preview:")
        print(r["text"][:preview_chars])


# =========================================================
# PROMPTING
# =========================================================
def build_macro_prompt(query, retrieved_chunks):
    context = "\n\n".join(
        [
            f"[Rank {r['rank']}] Section: {r['section']} | Subsection: {r['subsection']}\n{r['text']}"
            for r in retrieved_chunks
        ]
    )

    return f"""
You are a macro-financial risk analyst.

Your task is to write ONE clear paragraph explaining the current macroeconomic environment
and how it may affect household credit risk.

User query:
{query}

Retrieved context:
{context}

Rules:
1. Write exactly one paragraph.
2. Keep it between 90 and 150 words.
3. Focus only on macroeconomic conditions and their implications for household credit risk.
4. Mention only information supported by the retrieved context.
5. Do not output bullet points, JSON, or headings.
6. Use professional but simple language suitable for a dashboard.

Output only the paragraph.
""".strip()


# =========================================================
# OLLAMA
# =========================================================

def call_ollama(prompt: str, model_name: str = OLLAMA_MODEL_NAME) -> str:
    """Call Ollama local API."""
    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False,
    }

    response = requests.post(
        OLLAMA_URL,
        json=payload,
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()

    data = response.json()
    if "response" not in data:
        raise ValueError(f"Unexpected Ollama response format: {data}")

    return data["response"]


import re

def parse_json_output(output_text: str) -> list[dict[str, Any]] | None:
    """Parse model output into JSON, even if wrapped in markdown fences."""
    if not output_text:
        return None

    cleaned = output_text.strip()

    # Remove ```json ... ``` or ``` ... ```
    cleaned = re.sub(r"^```json\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    # Extract first JSON array if present
    match = re.search(r"(\[\s*{.*}\s*\])", cleaned, flags=re.DOTALL)
    if match:
        cleaned = match.group(1)

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        print("\n⚠ JSON parsing failed. Raw model output:\n")
        print(output_text)
        return None
    
def generate_macro_summary_year(year, embedding_model, index, df_meta, k=3):

    query = f"""
    Summarize the macroeconomic environment in {year} and explain how it affects household credit risk.
    """

    retrieved = retrieve_chunks_by_year(
        query, embedding_model, index, df_meta, year, k=k
    )

    prompt = build_macro_prompt(query, retrieved)
    output = call_ollama(prompt)

    return retrieved, output


# =========================================================
# FULL RAG AGENT
# =========================================================

def rag_agent(
    query: str,
    embedding_model: SentenceTransformer,
    index: Any,
    df_meta: pd.DataFrame,
    k: int = TOP_K,
) -> tuple[list[dict[str, Any]], str, list[dict[str, Any]] | None]:
    """Full pipeline: retrieve -> prompt -> ollama -> parse."""
    retrieved = retrieve_chunks(query, embedding_model, index, df_meta, k=k)
    prompt = build_prompt(query, retrieved)
    output_text = call_ollama(prompt)
    parsed = parse_json_output(output_text)
    return retrieved, output_text, parsed


# =========================================================
# MAIN
# =========================================================

def main() -> None:
    embedding_model, index, df_meta = load_resources()

    query = "Summarize the macroeconomic environment in 2008, especially labor market and credit conditions, and explain how it may influence household default risk."

    print("\nRunning retrieval...")
    retrieved, macro_paragraph = generate_macro_summary(
        query=query,
        embedding_model=embedding_model,
        index=index,
        df_meta=df_meta,
        k=3,
    )

    print("\nRetrieved chunks:")
    print_retrieval_results(retrieved, preview_chars=400)

    print("\n" + "#" * 100)
    print("MACRO ENVIRONMENT SUMMARY")
    print("#" * 100)
    print(macro_paragraph)


if __name__ == "__main__":
    main()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading FAISS index...
Loading metadata CSV...
Resources loaded successfully.

Running retrieval...

Retrieved chunks:

Rank: 1
Chunk ID: mpr_2012_07_chunk_007
Section: The Household Sector
Subsection: General
Distance: 0.9261
Text preview:
hours worked; real average hourly earnings are little changed thus far this year. The ratio of household net worth to income, in the aggregate, moved up slightly further in the first quarter, reflecting increases in both house prices and equity prices (figure 6). Taking a longer view, this ratio has been on a slow upward trend since 2009, and while it remains far below levels seen in the years lea

Rank: 2
Chunk ID: mpr_2023_03_chunk_035
Section: Financial Developments
Subsection: General
Distance: 0.9338
Text preview:
Part 1:  Recent Economic and Financial Developments remain elevated despite the rise in mortgage rates and sharply decelerating real estate prices, as the increase in house prices over the past two years has substantially exceeded the